In [31]:
%load_ext autoreload
%autoreload 2

In [34]:
import pandas as pd
import numpy as np
import mlflow
import joblib
from pathlib import Path
import os
import json
from keras.models import load_model
from src.model.model import custom_Net, modelNet
import sys
sys.path.append('..')

In [3]:
mlflow.set_tracking_uri(r"sqlite:///C:/Users/USER/Downloads/Projects/mode_choice/notebooks/mlflow.db")
experiment = mlflow.get_experiment_by_name("xgb_experiment")
print(experiment)

<Experiment: artifact_location='file:C:/Users/USER/Downloads/Projects/mode_choice/notebooks/mlruns/1', creation_time=1786811869937, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786811869937, lifecycle_stage='active', name='xgb_experiment', tags={}, trace_location=None, workspace='default'>


In [5]:
def get_best_fold_from_mlflow(experiment_name: str, metric_name: str = None):
    """
    Searches nested fold runs under the given experiment and finds the
    fold with the highest macro_pr_auc.
    """
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found")

    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    pr_auc_cols = [c for c in runs.columns if c.startswith("metrics.macro_pr_auc_")]

    if not pr_auc_cols:
        raise ValueError("No macro_pr_auc metrics found — check experiment name/logging.")

    best_fold = None
    best_score = -1
    best_run_id = None

    for col in pr_auc_cols:
        fold_idx = col.replace("metrics.macro_pr_auc_", "")
        
        col_vals = runs[["run_id", col]].dropna()
        if col_vals.empty:
            continue
        row = col_vals.loc[col_vals[col].idxmax()]
        score = row[col]
        if score > best_score:
            best_score = score
            best_fold = fold_idx
            best_run_id = row["run_id"]

    print(f"Best fold: {best_fold} | macro_pr_auc: {best_score:.4f} | run_id: {best_run_id}")
    return best_fold, best_score, best_run_id

In [6]:
best_fold_xgb, best_score_xgb, best_run_id_xgb = get_best_fold_from_mlflow("xgb_experiment")

Best fold: 8 | macro_pr_auc: 0.6853 | run_id: 16e1a1c3518342998a4ef1eaa0658e8d


In [10]:
mlflow.set_tracking_uri(r"sqlite:///C:/Users/USER/Downloads/Projects/mode_choice/notebooks/mlflow_tf.db")
experiment = mlflow.get_experiment_by_name("nn_experiment")
print(experiment)

<Experiment: artifact_location='/home/sagemaker-user/mode_choice_analysis/mlruns/1', creation_time=1786888839269, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786888839269, lifecycle_stage='active', name='nn_experiment', tags={}, trace_location=None, workspace='default'>


In [11]:
best_fold_nn, best_score_nn, best_run_id_nn = get_best_fold_from_mlflow("nn_experiment")

Best fold: 2 | macro_pr_auc: 0.6169 | run_id: d1d9246633bf4aac8732f0cb08391bf7


In [12]:
def test_xgb_model(X_test, y_test, model, thresholds):
    probs = model.predict_proba(X_test)
    adjusted = probs / thresholds 
    y_hat = np.argmax(adjusted, axis=1)
    confidence_score = np.max(probs, axis=1)

    report = classification_report(y_test,y_hat output_dict=True)
    cm = confusion_matrix(y_test, y_hat)

    return report, cm

In [13]:
def test_nn_model(X_test, y_test, model):
    probs = model.predict_proba(X_test)
    #adjusted = probs / thresholds 
    y_hat = np.argmax(probs, axis=1)
    confidence_score = np.max(probs, axis=1)

    report = classification_report(y_test, y_hat, output_dict=True)
    cm = confusion_matrix(y_test, y_hat)

    return report, cm

In [16]:
xgb_model = joblib.load(Path(os.getcwd())/'notebooks'/'xgboost_models'/'xgb_fold_8.joblib')

In [20]:
path = Path(os.getcwd())/'notebooks'/'xgboost_models'/'thresholds.json'
with open(path, 'r') as f:
    thresholds = json.load(f)

best_threshold = thresholds["8"]

In [27]:
model_dir = Path(r"C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts")
for p in model_dir.rglob("*"):
    print(p)

C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\class_weights_fold_2.json
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\model_summary.txt
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\tensorboard_logs
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint.h5
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint_metrics.json
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\tensorboard_logs\train
C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aa

In [26]:
print(model_dir.exists())
print(model_dir.is_dir())

False
False


In [29]:
checkpoint_path = r"C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint.h5"
print(os.path.getsize(checkpoint_path))

695176


In [28]:
model_dir = Path(r"C:\Users\USER\Downloads\Projects\
mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint.h5")
tf_model = load_model(model_dir)

OSError: [Errno 22] Unable to synchronously open file (unable to open file: name = 'C:\Users\USER\Downloads\Projects\
mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint.h5', errno = 22, error message = 'Invalid argument', flags = 0, o_flags = 0)

In [35]:
checkpoint_path = r"C:\Users\USER\Downloads\Projects\mode_choice\notebooks\mlruns\2\d1d9246633bf4aac8732f0cb08391bf7\artifacts\checkpoints\latest_checkpoint.h5"

tf_model = load_model(checkpoint_path, custom_objects={"modelNet": modelNet})

ValueError: Layer count mismatch when loading weights from file. Model expected 0 layers, found 6 saved layers.